# Découvrir les limites IGN de l’Auvergne-Rhône-Alpes

Ce notebook reprend les étapes déjà réalisées : télécharger les régions, examiner leur tableau, sélectionner notre région et afficher son contour.

Une **cellule Markdown** contient les explications. Une **cellule de code** exécute du Python avec `Maj + Entrée`. Exécuter les cellules dans l’ordre. Pour vérifier la reproductibilité, redémarrer le noyau puis tout exécuter.

Dans VS Code, sélectionner comme noyau `ign-osm/.venv/bin/python`, qui contient déjà GeoPandas et ipykernel. Si VS Code propose `/usr/bin/python3`, changer de noyau avant de lancer les cellules. Un noyau est le processus Python qui garde les variables entre les cellules.

Les communes et le réseau OSM seront traités lors des prochaines étapes.

## 1. Préparer les chemins

`Path` représente un chemin de fichier. GeoPandas manipule un tableau avec une colonne de formes géographiques. `display` affiche un tableau ou un dessin dans le notebook.

Un notebook n’a pas de variable `__file__` comme un script. Nous cherchons donc le dossier du module depuis le dossier courant ou ses parents. Cela fonctionne avec un lancement depuis la racine du dépôt, le module ou son dossier `notebooks`.

In [1]:
from pathlib import Path
from urllib.request import urlopen

import geopandas as gpd
from IPython.display import HTML, display

current_directory = Path.cwd().resolve()
module_directory = None

for directory in [current_directory, *current_directory.parents]:
    if directory.name == "ign-osm" and (directory / "requirements.txt").is_file():
        module_directory = directory
        break
    if (directory / "ign-osm" / "requirements.txt").is_file():
        module_directory = directory / "ign-osm"
        break

if module_directory is None:
    raise FileNotFoundError("Open the notebook from the project directory.")

raw_path = module_directory / "data" / "raw" / "ign" / "regions_2026.fgb"
region_path = module_directory / "data" / "processed" / "region_84.geojson"
preview_path = module_directory / "data" / "processed" / "region_84_preview.html"

print(f"Module directory: {module_directory}")

Module directory: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm


## 2. Récupérer les régions

Source : IGN **ADMIN EXPRESS COG**, édition du 1er janvier 2026, issue de la source ADMIN EXPRESS indiquée dans le sujet.

- [Présentation officielle IGN](https://cartes.gouv.fr/aide/fr/partenaires/ign/generalites-ign/actualites/2026-06-flatgeobuf-geoparquet/)
- Les informations de récupération sont conservées dans `../sources.csv`.
- La licence reste à confirmer dans les métadonnées officielles avant redistribution.

Le fichier `.fgb` (FlatGeobuf) contient les contours et leurs attributs. Ce format binaire ne se lit pas comme du texte.

Si le fichier existe, nous le réutilisons pour éviter un téléchargement à chaque exécution. Sinon, cette cellule nécessite Internet. Les premiers octets, appelés **signature**, permettent de vérifier le format ; ils ne vérifient pas la précision des contours.

In [2]:
source_url = (
    "https://data.geopf.fr/chunk/telechargement/download/"
    "ADMIN-EXPRESS-COG-PARTIEL/"
    "ADMIN-EXPRESS-COG_4-0__FLATGEOBUF_WGS84G_FRA_2026-01-01/region.fgb"
)

if raw_path.exists():
    print(f"Using existing file: {raw_path}")
else:
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(source_url, timeout=120) as response:
        content = response.read()
    if not content.startswith(b"fgb\x03fgb\x01"):
        raise ValueError("The response is not a FlatGeobuf file.")
    raw_path.write_bytes(content)
    print(f"Downloaded {len(content):,} bytes")

with raw_path.open("rb") as source_file:
    if source_file.read(8) != b"fgb\x03fgb\x01":
        raise ValueError("The local file is not a FlatGeobuf file.")

Using existing file: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/raw/ign/regions_2026.fgb


## 3. Explorer le tableau

`read_file()` ouvre le fichier. Chaque ligne représente une région. `geometry` contient son contour ; nous masquons cette colonne dans l’affichage pour lire les autres attributs.

Les noms des colonnes ci-dessous sont ceux de l’IGN, conservés tels quels à la lecture. Nous les renommerons en anglais dans notre sortie.

Le **CRS** est le système qui donne un sens aux coordonnées. EPSG:4326 signifie ici longitude et latitude en degrés : on ne peut pas y mesurer directement des distances en mètres.

In [3]:
regions = gpd.read_file(raw_path)
print(f"Region count: {len(regions)}")
print(f"Coordinate reference system: {regions.crs}")
display(regions.drop(columns="geometry"))

Region count: 18
Coordinate reference system: EPSG:4326


,cleabs,nom_officiel,nom_officiel_en_majuscules,code_insee,code_siren
0,REGION__0000000000000004,La Réunion,LA REUNION,04,179740014
1,REGION__0000000000000006,Mayotte,MAYOTTE,06,130003346
2,REGION__0000000000000094,Corse,CORSE,94,172010001
3,REGION__0000000000000093,Provence-Alpes-Côte d'Azur,PROVENCE-ALPES-COTE D'AZUR,93,171300007
4,REGION__0000000000000084,Auvergne-Rhône-Alpes,AUVERGNE-RHONE-ALPES,84,176900009
5,REGION__0000000000000044,Grand Est,GRAND EST,44,176700003
6,REGION__0000000000000027,Bourgogne-Franche-Comté,BOURGOGNE-FRANCHE-COMTE,27,172100000
7,REGION__0000000000000024,Centre-Val de Loire,CENTRE-VAL DE LOIRE,24,174500009
8,REGION__0000000000000011,Île-de-France,ILE-DE-FRANCE,11,177500006
9,REGION__0000000000000032,Hauts-de-France,HAUTS-DE-FRANCE,32,175900000


## 4. Sélectionner la région

Le code INSEE est un identifiant administratif conservé comme du texte. Pour l’Auvergne-Rhône-Alpes, il vaut `84`.

`.loc` sélectionne les lignes et les colonnes, comme un `WHERE` et un `SELECT` SQL. `.copy()` crée un tableau indépendant. Nous exigeons exactement une ligne avec une géométrie non vide et valide, puis renommons les colonnes utiles en anglais.

In [4]:
region_code = "84"
region = regions.loc[
    regions["code_insee"] == region_code,
    ["code_insee", "nom_officiel", "geometry"],
].copy()

if len(region) != 1:
    raise ValueError("Expected exactly one region with INSEE code 84.")
if region.geometry.is_empty.any() or not region.geometry.is_valid.all():
    raise ValueError("The region boundary is empty or invalid.")

region = region.rename(
    columns={"code_insee": "insee_code", "nom_officiel": "name"}
)
region = region.to_crs(epsg=4326)
display(region[["insee_code", "name"]])

,insee_code,name
4,84,Auvergne-Rhône-Alpes


## 5. Enregistrer et relire le résultat

Le **GeoJSON** est un format JSON contenant des objets géographiques. Nous sauvegardons une version exploitable par l’équipe, puis la relisons pour vérifier le fichier réellement écrit. Relancer la cellule remplace cette sortie, mais ne modifie pas la source IGN.

In [5]:
region_path.parent.mkdir(parents=True, exist_ok=True)
region.to_file(region_path, driver="GeoJSON")

saved_region = gpd.read_file(region_path)
assert len(saved_region) == 1
assert saved_region.iloc[0]["insee_code"] == region_code
assert saved_region.geometry.is_valid.all()
print(f"Saved and verified: {region_path}")

Saved and verified: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/processed/region_84.geojson


## 6. Afficher le contour

Le **SVG** est un dessin vectoriel que le navigateur peut afficher. Nous réutilisons la génération HTML existante et affichons son résultat directement ici.

La projection **Lambert-93** (EPSG:2154) exprime les coordonnées en mètres et convient à la France métropolitaine. `simplify(100)` allège uniquement le dessin avec une tolérance de 100 mètres ; le GeoJSON n’est pas modifié. `bounds` permet de cadrer le dessin. L’axe vertical du SVG est inversé pour garder le nord en haut.

Cette vue n’a pas de fond de carte : elle montre uniquement le contour régional.

In [6]:
preview_region = gpd.read_file(region_path)

# Use a metric coordinate system to preserve the region's proportions.
preview_region = preview_region.to_crs(epsg=2154)
boundary = preview_region.geometry.iloc[0].simplify(100, preserve_topology=True)

min_x, min_y, max_x, max_y = boundary.bounds
padding = 10000
width = max_x - min_x + 2 * padding
height = max_y - min_y + 2 * padding
view_box = f"{min_x - padding} {-max_y - padding} {width} {height}"

# SVG coordinates increase downward; flip the geometry to keep north up.
boundary_svg = boundary.svg(fill_color="#2563eb", opacity=0.6)

html = f"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Auvergne-Rhône-Alpes boundary</title>
  <style>
body {{ font-family: sans-serif; max-width: 900px; margin: 32px auto; padding: 0 16px; }}
svg {{ width: 100%; height: 65vh; background: #f1f5f9; border-radius: 12px; }}
path {{ stroke: #1e40af; stroke-width: 600; }}
  </style>
</head>
<body>
  <h1>Auvergne-Rhône-Alpes</h1>
  <p>INSEE code: 84 | North is up | IGN ADMIN EXPRESS COG 2026</p>
  <svg xmlns="http://www.w3.org/2000/svg" viewBox="{view_box}"
   role="img" aria-label="Auvergne-Rhône-Alpes region boundary">
<g transform="scale(1,-1)">{boundary_svg}</g>
  </svg>
  <p>Display simplified to 100 metres. Original GeoJSON unchanged.</p>
  <p>Boundary preview only: no streets, stations or population data.</p>
</body>
</html>
"""
preview_path.write_text(html, encoding="utf-8")
print(f"Saved preview to {preview_path}")


display(HTML(html))

Saved preview to /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/ign-osm/data/processed/region_84_preview.html


## Bilan

Nous avons récupéré la source IGN, exploré ses attributs, sélectionné le code `84`, exporté puis relu le GeoJSON et affiché son contour.

Fichiers produits dans `ign-osm/data/processed/` :
- `region_84.geojson` : contour original sélectionné, pour les traitements ;
- `region_84_preview.html` : aperçu simplifié, consultable hors connexion.

La validité géométrique et l’affichage ne prouvent pas la précision indépendante de la frontière. Le fichier ne contient ni habitants, ni stations, ni communes.

**Prochaine étape : récupérer les communes de cette région.**